# Test path classification model

Initialize the device to use for model inference

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("is cuda available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Initialize data paths and split that were used during training

In [ ]:
import os

data_dir = os.path.abspath('../data/FIVES')
train_split = 'train_clean'

Initialize the model components with the same hyperparameters as during training

In [ ]:
from path_neural_networks.models.features_generators import FeaturesGenerator, PretrainedUnetFeaturesGenerator
from path_neural_networks.models.path_samplers import PathSampler, MultiScaleSquarePathSampling, SamplingMaxAggregation
from path_neural_networks.models.path_encoders import PathEncoder, ConvMaxPoolingPathEncoder
from path_neural_networks.models.path_classifiers import PathClassifier, FCNPathClassifier
from utils.other import pretty_dict_print

unet_checkpoint_dir = os.path.abspath('../checkpoints/unet_pretraining')
unet_ckpt_path = os.path.join(unet_checkpoint_dir, os.listdir(unet_checkpoint_dir)[0])
print("Using UNet checkpoint:", unet_ckpt_path)

features_generator_out_channels = 32
sampling_square_sizes = [1, 3, 5]
sampling_aggregation_method = SamplingMaxAggregation()
conv_path_residual_blocks = False
conv_path_skip_connections = False
conv_path_layers = [None, None, 256]
path_classifier_n_hidden_layers = 2
path_classifier_dropout = 0

features_generator: FeaturesGenerator = PretrainedUnetFeaturesGenerator(
    ckpt_path=unet_ckpt_path,
    device=device,
    out_channels=features_generator_out_channels,
    freeze_pretrained=False,
    skip_connection=False
)
path_sampler: PathSampler = MultiScaleSquarePathSampling(
    in_channels=features_generator_out_channels,
    square_sizes=sampling_square_sizes,
    aggregation=sampling_aggregation_method
)
path_encoder: PathEncoder = ConvMaxPoolingPathEncoder(
    in_channels=path_sampler.out_channels, 
    hidden_layers=conv_path_layers, 
    skip_connection=conv_path_skip_connections, 
    residual_blocks=conv_path_residual_blocks
)
path_classifier: PathClassifier = FCNPathClassifier(
    in_channels=path_encoder.out_channels,
    n_hidden_layers=path_classifier_n_hidden_layers,
    num_classes=1,
    dropout=path_classifier_dropout
)

Initialize datamodules

In [ ]:
from path_neural_networks.data.image_centerline_dataset import ImageCenterlineDataset
from path_neural_networks.data.image_centerline_datamodule import ImageCenterlineDatamodule
import albumentations as A
from albumentations.pytorch import ToTensorV2

max_dist = 100

if max_dist is None:
    centerlines_dirname = "euclidean_all_centerlines"
else:
    centerlines_dirname = f"euclidean_lt_{max_dist}_centerlines"

split_file_path=os.path.join(data_dir, "splits.json")
val_split_ratio = 0.2
use_foreground_pixels_only_for_normalization = True
data_seed = 42
split_seed = 42

dataset = ImageCenterlineDataset(data_dir=data_dir, centerline_dirname=centerlines_dirname)
datamodule = ImageCenterlineDatamodule(dataset=dataset, 
                                        split_file_path=split_file_path,
                                        train_split_name=train_split,
                                        val_split_ratio=val_split_ratio,
                                        seed = split_seed)
datamodule.setup()

stats = dataset.get_dataset_stats(split_name='train_clean', split_indices=datamodule.train_indices.tolist() + datamodule.val_indices.tolist())
if use_foreground_pixels_only_for_normalization:
    stats = stats['foreground']
else:
    stats = stats['full_image']
mean, std = stats['mean'], stats['std']
print("Dataset stats used for normalization:")
pretty_dict_print(stats)

test_transforms = A.Compose([
    A.Normalize(mean=mean, std=std, max_pixel_value=1.0),
    ToTensorV2()
], seed=data_seed)

datamodule = ImageCenterlineDatamodule(dataset=dataset,
                                        split_file_path=split_file_path,
                                        train_split_name=train_split,
                                        val_split_ratio=val_split_ratio,
                                        test_transforms=test_transforms,
                                        seed = split_seed)

Initialize the loss function

In [ ]:
from path_neural_networks.models.losses import PathClassificationLoss, WeightedBCEWithLogitsLoss, BCEWithLogitsLoss

use_pos_weight_in_loss = True

loss_fn: PathClassificationLoss
if use_pos_weight_in_loss:
    classes_stats = dataset.get_dataset_classes_stats()
    classes_ratio = classes_stats['classes_ratio']
    loss_fn = WeightedBCEWithLogitsLoss(classes_ratio=classes_ratio)
else:
    loss_fn = BCEWithLogitsLoss()

## Initialize the model checkpoint, two choices:

### Case #1: Use the provided weights (see [README](../README.md)) to initialize the pretrained path classification model

In [ ]:
checkpoint_dir = os.path.abspath('../checkpoints/main_model_pretrained')
ckpt_path = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])
print("Using UNet checkpoint:", ckpt_path)

### Case #2: Use the model trained in the [preceding notebook (5)](./05_train_path_classification_model.ipynb)

In [ ]:
checkpoint_dir = os.path.abspath('../checkpoints/main_model_training')
ckpt_path = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])
print("Using UNet checkpoint:", ckpt_path)

## Load model and trainer

In [ ]:
from path_neural_networks.models import ReducedPipelineLitModule
from path_neural_networks.utils.symmetry_enforcement import SymmetryEnforcementMode

metrics = ["accuracy", "auroc", "recall", "precision", "pr_auc"]
symmetry_enforcement_mode = SymmetryEnforcementMode.NONE

model = ReducedPipelineLitModule.load_from_checkpoint(
    ckpt_path,
    features_generator=features_generator,
    path_sampler=path_sampler,
    path_encoder=path_encoder,
    path_classifier=path_classifier,
    edge_classification_loss_fn=loss_fn,
    metrics=metrics,
    symmetry_enforcement_mode=symmetry_enforcement_mode
)

In [ ]:
from pytorch_lightning import Trainer

torch.set_float32_matmul_precision("medium")
trainer = Trainer(accelerator='gpu', devices="auto", max_epochs=10, precision='16-mixed')

## Testing the trained model

In [ ]:
trainer.test(model, datamodule=datamodule)

### Finding the best threshold on the validation set

In [ ]:
from tqdm import tqdm

def perform_inference(model, dataloader, apply_sigmoid: bool = True):
    all_preds = []
    all_targets = []

    model.eval()
    with torch.no_grad():
        for batch in tqdm(dataloader):
            img, (paths, edges_classes), _ = batch
            path_logits, _ = model(img, paths)
            path_probs = torch.sigmoid(path_logits)
            all_preds.append(path_probs.detach().cpu())
            assert path_probs.shape[0] == edges_classes.shape[1], f"Number of edge predictions ({path_probs.shape[0]}) does not match number of edge labels ({edges_classes.shape[1]})"
            all_targets.append(edges_classes.squeeze().detach().cpu())

    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    if apply_sigmoid:
        all_preds = torch.sigmoid(all_preds)

    print(f"Performed inference on {len(dataloader)} cases, and on a total of {all_preds.shape[0]} paths.")

    return all_preds, all_targets

In [ ]:
preds, targets = perform_inference(model, datamodule.val_dataloader())

We compute the threshold that maximizes the F1 score on the validation set, to use it later for inference on the test set

In [ ]:
from torchmetrics.classification import BinaryPrecisionRecallCurve

pr_curve = BinaryPrecisionRecallCurve()
precision, recall, thresholds = pr_curve(preds, targets)

max_f1_threshold = thresholds[torch.argmax(2 * (precision * recall) / (precision + recall + 1e-8))]

print(max_f1_threshold)

## Quantitative analysis

In [ ]:
model.set_inference_threshold(max_f1_threshold.item())
preds, targets = perform_inference(model, datamodule.test_dataloader())

As it was done in the article, we compare our trained model to a naive model, based on the distance between the endpoints to connect, without any image-based features.

### Naive model initialization and inference

We first want to find the optimal distance threshold for this naive model

In [ ]:
from path_neural_networks.models.naive_model import NaiveModelLitModule
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import matplotlib.pyplot as plt

accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

x_range = np.arange(0, 105, 5)

for dist_threshold in x_range:
    naive_model = NaiveModelLitModule(max_dist=dist_threshold, metrics=metrics)

    naive_preds, naive_targets = perform_inference(naive_model, datamodule.val_dataloader(), apply_sigmoid=False)

    naive_accuracy = accuracy_score(naive_targets.numpy(), naive_preds.numpy())
    naive_precision = precision_score(naive_targets.numpy(), naive_preds.numpy(), zero_division=1.0)
    naive_recall = recall_score(naive_targets.numpy(), naive_preds.numpy())
    naive_f1 = f1_score(naive_targets.numpy(), naive_preds.numpy())

    accuracy_scores.append(naive_accuracy)
    precision_scores.append(naive_precision)
    recall_scores.append(naive_recall)
    f1_scores.append(naive_f1)

naive_max_f1_threshold_index = np.argmax(f1_scores)
naive_max_f1_threshold = x_range[naive_max_f1_threshold_index]

plt.plot(x_range, accuracy_scores, label="Accuracy")
plt.plot(x_range, precision_scores, label="Precision")
plt.plot(x_range, recall_scores, label="Recall")
plt.plot(x_range, f1_scores, label="F1 Score")
plt.axvline(x=naive_max_f1_threshold, color='r', linestyle='--', label=f'Max F1 - Distance Threshold: {naive_max_f1_threshold}')
plt.xlabel("Distance Threshold")
plt.ylabel("Score")
plt.title("Naive Model Performance vs Distance Threshold")
plt.legend()
plt.grid()
plt.show()